# Q-learning Volleyball

Attach: `volleyball_game.zip`

## Watch Before Training

In [ ]:
GAME_PASSWORD = "000"

import warnings

warnings.filterwarnings(
    "ignore",
    message=r"no fc_cache font cache file.*",
    category=UserWarning,
)


In [ ]:
import math
from pathlib import Path
import random
import re
import sys
import zipfile

import numpy as np

import pygame
from PIL import Image as PILImage

zip_candidates = (
    Path("uploads/volleyball_game.zip"),
    Path("volleyball_game.zip"),
    Path("/content/volleyball_game.zip"),
)
game_zip = next((path for path in zip_candidates if path.exists()), None)
if game_zip is None:
    raise FileNotFoundError("Attach volleyball_game.zip before running this cell.")

runtime_root = Path(".volleyball_game_runtime")
runtime_root.mkdir(exist_ok=True)
with zipfile.ZipFile(game_zip) as archive:
    archive.extractall(runtime_root)

conf_path = runtime_root / "_10_config" / "conf.py"


def apply_game_password():
    conf_text = conf_path.read_text(encoding="utf-8")
    conf_text = re.sub(
        r"^\s*BNW_MODE_PW\s*=.*$",
        f'BNW_MODE_PW = "{GAME_PASSWORD}"',
        conf_text,
        flags=re.MULTILINE,
    )
    conf_path.write_text(conf_text, encoding="utf-8")


apply_game_password()

runtime_path = str(runtime_root.resolve())
if runtime_path not in sys.path:
    sys.path.insert(0, runtime_path)
for module_name in list(sys.modules):
    if module_name == "_00_environment" or module_name.startswith("_00_environment."):
        del sys.modules[module_name]

from _00_environment import Env
from _00_environment.input import UserInput

ACTION_NAMES = (
    "IDLE",
    "LEFT",
    "RIGHT",
    "JUMP",
    "JUMP_LEFT",
    "JUMP_RIGHT",
    "DIVE_LEFT",
    "HIT_UP",
    "HIT_FLAT",
    "HIT_DOWN",
)
RAW_ACTIONS = (
    (0, 0, 0),
    (-1, 0, 0),
    (1, 0, 0),
    (0, -1, 0),
    (-1, -1, 0),
    (1, -1, 0),
    (-1, 0, 1),
    (1, -1, 1),
    (1, 0, 1),
    (1, 1, 1),
)
ACTION_REPEAT = 4
EPISODES_PER_BLOCK = 800
RANDOM_SEED = 100
EVALUATION_SEED = 7100


def raw_input(action_index):
    x_direction, y_direction, power_hit = RAW_ACTIONS[action_index]
    user_input = UserInput()
    user_input.x_direction = x_direction
    user_input.y_direction = y_direction
    user_input.power_hit = power_hit
    return user_input


def bin_by_edges(value, edges):
    return sum(value > edge for edge in edges)


def compact_state(env):
    engine = env.engine
    player = engine.players[0]
    ball = engine.ball
    engine.update_expected_landing_point()
    return (
        int(ball.x > 216),
        bin_by_edges(ball.expected_landing_point_x - player.x, (-70, -20, 20, 70)),
        bin_by_edges(ball.x - player.x, (-70, -20, 20, 70)),
        bin_by_edges(ball.y, (90, 180)),
        int(ball.y_velocity > 0),
        0 if player.state == 0 else 1 if player.state in (1, 2) else 2,
    )


def defensive_distance(env):
    env.engine.update_expected_landing_point()
    player_x = env.engine.players[0].x
    landing = env.engine.ball.expected_landing_point_x
    target = landing if 20 <= landing <= 196 else 108
    return abs(target - player_x)


def raw_decision_step(env, action_index):
    total_reward = 0.0
    discount = 1.0
    done = False
    last_score = None
    for _ in range(ACTION_REPEAT):
        before_distance = defensive_distance(env)
        score, _, _, rewards = env.run_training_step(
            "player1",
            raw_input(action_index),
            opponent="rule",
        )
        after_distance = defensive_distance(env)
        events = score.get("events") or {}
        reward = 3.0 * float(rewards["player1"])
        reward += 0.03 * max(
            -30.0,
            min(30.0, before_distance - after_distance),
        )
        if (events.get("touch") or {}).get("player1"):
            reward += 5.0
        if events.get("crossed_net_to") == "player2":
            reward += 2.0
        total_reward += discount * reward
        discount *= 0.95
        last_score = score
        if score["match_done"]:
            done = True
            break
    return total_reward, done, last_score


def q_values(state):
    if state not in q_table:
        q_table[state] = np.zeros(len(RAW_ACTIONS), dtype=np.float32)
    return q_table[state]


def greedy_action(policy, state, rng):
    values = policy.get(state)
    if values is None:
        return rng.randrange(len(RAW_ACTIONS))
    candidates = np.flatnonzero(values == values.max())
    return int(rng.choice(candidates))


def print_progress(completed_episodes):
    width = 30
    filled = round(width * completed_episodes / EPISODES_PER_BLOCK)
    percent = round(100 * completed_episodes / EPISODES_PER_BLOCK)
    print(f"[{'#' * filled}{'-' * (width - filled)}] {percent:3d}%", flush=True)


def evaluate(policy, matches=300):
    python_random_state = random.getstate()
    numpy_random_state = np.random.get_state()
    rng = random.Random(EVALUATION_SEED + 1)
    wins = 0
    try:
        random.seed(EVALUATION_SEED)
        np.random.seed(EVALUATION_SEED)
        env = Env(
            render_mode="log",
            target_score=1,
            seed=EVALUATION_SEED,
            randomize_serve_on_reset=True,
            rally_step_limit=1200,
        )
        for _ in range(matches):
            env.reset(randomize_serve=True, return_state=False)
            while True:
                action_index = greedy_action(policy, compact_state(env), rng)
                _, done, score = raw_decision_step(env, action_index)
                if done:
                    wins += int(score["player1"] > score["player2"])
                    break
        env.close()
    finally:
        random.setstate(python_random_state)
        np.random.set_state(numpy_random_state)
    return wins / matches


def train_episode_block():
    global training_step, training_episode
    block = training_episode // EPISODES_PER_BLOCK + 1
    first_episode = training_episode + 1
    final_episode = training_episode + EPISODES_PER_BLOCK
    random.seed(RANDOM_SEED + block)
    np.random.seed(RANDOM_SEED + block)
    print(f"\nTRAINING EPISODES {first_episode}-{final_episode}")
    print_progress(0)

    for local_episode in range(1, EPISODES_PER_BLOCK + 1):
        while True:
            state = compact_state(training_env)
            values = q_values(state)
            total_steps = training_step
            training_step += 1
            epsilon = max(0.03, 0.95 * math.exp(-total_steps / 48000))
            if training_rng.random() < epsilon:
                action_index = training_rng.randrange(len(RAW_ACTIONS))
            else:
                candidates = np.flatnonzero(values == values.max())
                action_index = int(training_rng.choice(candidates))

            reward, done, _ = raw_decision_step(training_env, action_index)
            next_state = compact_state(training_env)
            next_values = q_values(next_state)
            target = reward if done else reward + 0.98 * float(next_values.max())
            alpha = max(0.02, 0.12 * math.exp(-total_steps / 60000))
            values[action_index] += alpha * (
                target - float(values[action_index])
            )
            episode_trajectory.append((state, action_index, reward))

            if done:
                episode_return = 0.0
                for old_state, old_action, old_reward in reversed(episode_trajectory):
                    episode_return = old_reward + 0.995 * episode_return
                    old_values = q_table[old_state]
                    old_values[old_action] += 0.8 * alpha * (
                        episode_return - float(old_values[old_action])
                    )
                episode_trajectory.clear()
                training_env.reset(randomize_serve=True, return_state=False)
                training_episode += 1
                break

        if local_episode % (EPISODES_PER_BLOCK // 20) == 0:
            print_progress(local_episode)

    win_rate = evaluate(q_table)
    print(
        f"RESULT | {training_episode} EPISODES | WIN RATE {win_rate:.1%} | "
        f"Q-STATES {len(q_table)}/900"
    )
    return win_rate


class RawQAgent:
    def __init__(self, policy, learned_episodes):
        self.policy = {state: values.copy() for state, values in policy.items()}
        self.policy_name = f"{learned_episodes}ep raw Q"
        self.rng = random.Random(8000 + learned_episodes)
        self.env = None
        self.action_index = 0
        self.frames_left = 0

    def select_action(self, state):
        if self.frames_left <= 0 or self.env.rally_step_count == 0:
            compact = compact_state(self.env)
            self.action_index = greedy_action(self.policy, compact, self.rng)
            self.frames_left = ACTION_REPEAT
        self.frames_left -= 1
        return raw_input(self.action_index)


def play_against_snapshot(learned_episodes):
    apply_game_password()
    agent = RawQAgent(q_table, learned_episodes)
    env = Env(
        render_mode="human",
        target_score=5,
        seed=2000 + learned_episodes // EPISODES_PER_BLOCK,
        randomize_serve_on_reset=True,
        rally_step_limit=3000,
    )
    env.set(player1=agent, player2="rule", random_serve=True, return_state=False)
    started_bnw = bool(env.engine.viewer.bnw_mode)
    print(f"WATCH | {learned_episodes} EPISODES RAW Q VS RULE", flush=True)
    last_result = {"score": {"p1": 0, "p2": 0}, "done": False}
    while True:
        last_result = env.get_play_result()
        if last_result["done"] or env.engine.viewer.closed_requested:
            break
    result = {
        "score": dict(last_result["score"]),
        "started_bnw": started_bnw,
        "ended_bnw": bool(env.engine.viewer.bnw_mode),
    }
    env.close()
    print(f"SCORE {result['score']['p1']} : {result['score']['p2']}")
    return result


q_table = {}
episode_trajectory = []
training_rng = random.Random(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
training_env = Env(
    render_mode="log",
    target_score=1,
    seed=RANDOM_SEED,
    randomize_serve_on_reset=True,
    rally_step_limit=1200,
)
training_env.reset(randomize_serve=True, return_state=False)
training_step = 0
training_episode = 0


In [ ]:
battle_0ep = play_against_snapshot(0)

## Train Episodes 1-800

In [ ]:
score_800ep = train_episode_block()

## Watch Q-learning AI vs Rule AI

In [ ]:
battle_800ep = play_against_snapshot(800)

## Train 800 More Episodes (801-1600)

In [ ]:
score_1600ep = train_episode_block()

## Watch Q-learning AI vs Rule AI

In [ ]:
battle_1600ep = play_against_snapshot(1600)

## Train 800 More Episodes (1601-2400)

In [ ]:
score_2400ep = train_episode_block()

## Watch Q-learning AI vs Rule AI

In [ ]:
battle_2400ep = play_against_snapshot(2400)

## Train 800 More Episodes (2401-3200)

In [ ]:
score_3200ep = train_episode_block()

## Watch Q-learning AI vs Rule AI

In [ ]:
battle_3200ep = play_against_snapshot(3200)

## Train 800 More Episodes (3201-4000)

In [ ]:
score_4000ep = train_episode_block()

## Watch Q-learning AI vs Rule AI

In [ ]:
battle_4000ep = play_against_snapshot(4000)